# 03 - Model Experiments
**Credit Score Classification | MLOps Group 8**

This notebook visualises and compares all model training results:
- Load ranking artifacts produced by the training pipeline
- Compare 5 base models on macro F1, weighted F1, accuracy, and AUC
- Compare base vs. tuned vs. ensemble performance
- Confusion matrices for each model
- Calibration curves
- ROC curves (one-vs-rest)
- Learning curves for top model

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

REPO_ROOT   = Path().resolve().parent
REPORTS_DIR = REPO_ROOT / 'artifacts' / 'reports'
MODELS_DIR  = REPO_ROOT / 'artifacts' / 'models'
FIGURES_DIR = REPORTS_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

import sys
sys.path.insert(0, str(REPO_ROOT))

CLASS_NAMES = ['Poor', 'Standard', 'Good']
COLOR_MAP   = {'Poor': '#d9534f', 'Standard': '#f0ad4e', 'Good': '#5cb85c'}
MODEL_COLORS = ['#4e79a7','#f28e2b','#e15759','#76b7b2','#59a14f']

print('Repo root:', REPO_ROOT)

## 1. Load Ranking Artifacts

In [ ]:
ranking_path = REPORTS_DIR / 'model_ranking.csv'
if not ranking_path.exists():
    raise FileNotFoundError('Run training_pipeline first to generate model artifacts.')

ranking_df = pd.read_csv(ranking_path)
print('Model Ranking:')
ranking_df

In [ ]:
# Load additional report files if they exist
raw_comp_path = REPORTS_DIR / 'raw_model_comparison.csv'
tuning_path   = REPORTS_DIR / 'top3_tuning_results.csv'
final_path    = REPORTS_DIR / 'final_model_comparison.csv'

raw_comp_df = pd.read_csv(raw_comp_path) if raw_comp_path.exists() else None
tuning_df   = pd.read_csv(tuning_path)   if tuning_path.exists()   else None
final_df    = pd.read_csv(final_path)    if final_path.exists()    else None

print(f'raw_model_comparison loaded: {raw_comp_df is not None}')
print(f'top3_tuning_results loaded:  {tuning_df is not None}')
print(f'final_model_comparison loaded:{final_df is not None}')

## 2. Base Model Comparison

In [ ]:
metrics = ['f1_macro', 'f1_weighted', 'accuracy']
if 'auc_ovr' in ranking_df.columns:
    metrics.append('auc_ovr')

fig, axes = plt.subplots(1, len(metrics), figsize=(5 * len(metrics), 6))

for ax, metric in zip(axes, metrics):
    data = ranking_df.sort_values(metric, ascending=True)
    bars = ax.barh(data['model_name'], data[metric],
                   color=MODEL_COLORS[:len(data)], edgecolor='white')
    for bar, val in zip(bars, data[metric]):
        ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
                f'{val:.4f}', va='center', fontsize=9)
    ax.set_xlabel(metric)
    ax.set_title(metric.replace('_', ' ').title())
    ax.set_xlim(0, 1)

plt.suptitle('Base Model Comparison', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison_base.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Tuning Results (Top-3 Models)

In [ ]:
if tuning_df is not None:
    print('Tuning Results:')
    display_cols = [c for c in ['model_name','base_f1_macro','tuned_f1_macro',
                                'improvement','n_trials','best_params'] if c in tuning_df.columns]
    print(tuning_df[display_cols].to_string(index=False))
else:
    print('Tuning results not yet generated.')

In [ ]:
if tuning_df is not None and 'base_f1_macro' in tuning_df.columns and 'tuned_f1_macro' in tuning_df.columns:
    fig, ax = plt.subplots(figsize=(8, 5))
    x = np.arange(len(tuning_df))
    w = 0.35
    ax.bar(x - w/2, tuning_df['base_f1_macro'],  width=w, label='Base',  color='#4e79a7', edgecolor='white')
    ax.bar(x + w/2, tuning_df['tuned_f1_macro'], width=w, label='Tuned', color='#f28e2b', edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(tuning_df['model_name'], rotation=15)
    ax.set_ylabel('Macro F1')
    ax.set_title('Base vs Tuned Macro F1 (Top-3)')
    ax.legend()
    ax.set_ylim(0, 1)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'tuning_comparison.png', dpi=150)
    plt.show()

## 4. Final Model Comparison (Base vs Tuned vs Ensemble)

In [ ]:
if final_df is not None:
    print('Final Model Comparison:')
    print(final_df.to_string(index=False))

    if 'f1_macro' in final_df.columns and 'stage' in final_df.columns:
        fig, ax = plt.subplots(figsize=(10, 5))
        pivot = final_df.pivot(index='model_name', columns='stage', values='f1_macro')
        pivot.plot(kind='bar', ax=ax, edgecolor='white')
        ax.set_title('Macro F1 by Stage (Base / Tuned / Ensemble / Calibrated)')
        ax.set_ylabel('Macro F1')
        ax.set_ylim(0, 1)
        ax.set_xticklabels(ax.get_xticklabels(), rotation=15)
        ax.legend(title='Stage')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'final_model_comparison.png', dpi=150)
        plt.show()
else:
    print('Final model comparison not yet generated.')

## 5. Confusion Matrices

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Load confusion matrix CSVs if they exist
cm_files = list(FIGURES_DIR.glob('cm_*.png'))
print(f'Confusion matrix images found: {len(cm_files)}')
for f in cm_files:
    print(' ', f.name)

In [ ]:
# Load final model and run predictions if bundle exists
bundle_path = MODELS_DIR / 'final_model_bundle.pkl'
data_path   = REPO_ROOT / 'data' / 'processed' / 'test.csv'

if bundle_path.exists() and data_path.exists():
    from src.models.serialize import load_bundle

    bundle = load_bundle(bundle_path)
    df_test = pd.read_csv(data_path)

    # Use bundle.transform for full preprocessing
    X_test = bundle.transform(df_test)
    y_test = df_test['Credit_Score'].map({'Poor': 0, 'Standard': 1, 'Good': 2}).values

    # Filter to rows where y_test is not NaN (should be all since test came from train_raw)
    mask = ~np.isnan(y_test.astype(float))
    y_test = y_test[mask].astype(int)
    X_test = X_test[mask]

    y_pred = bundle.model.predict(X_test)

    cm = confusion_matrix(y_test, y_pred)
    fig, ax = plt.subplots(figsize=(7, 6))
    disp = ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES)
    disp.plot(ax=ax, cmap='Blues', colorbar=False)
    ax.set_title('Final Model -- Confusion Matrix (Test Set)', fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'cm_final_model_test.png', dpi=150)
    plt.show()
    print('Confusion matrix saved.')
else:
    print('Model bundle or test data not yet available.')

## 6. ROC Curves (One-vs-Rest)

In [ ]:
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize

if bundle_path.exists() and data_path.exists():
    y_proba = bundle.model.predict_proba(X_test)
    y_bin   = label_binarize(y_test, classes=[0, 1, 2])

    fig, ax = plt.subplots(figsize=(8, 6))
    for i, cls in enumerate(CLASS_NAMES):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_proba[:, i])
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, label=f'{cls} (AUC={roc_auc:.3f})',
                color=list(COLOR_MAP.values())[i], linewidth=2)

    ax.plot([0,1],[0,1], 'k--', alpha=0.5)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title('ROC Curves (One-vs-Rest) -- Final Model', fontsize=13)
    ax.legend(loc='lower right')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'roc_curves_final.png', dpi=150)
    plt.show()
else:
    print('Model bundle not yet available.')

## 7. Calibration Curves

In [ ]:
from sklearn.calibration import calibration_curve

if bundle_path.exists() and data_path.exists():
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for i, (cls, ax) in enumerate(zip(CLASS_NAMES, axes)):
        prob_true, prob_pred = calibration_curve(
            (y_test == i).astype(int), y_proba[:, i],
            n_bins=10, strategy='uniform'
        )
        ax.plot(prob_pred, prob_true, 'o-', label='Model',
                color=list(COLOR_MAP.values())[i])
        ax.plot([0,1],[0,1],'k--', alpha=0.5, label='Perfect')
        ax.set_title(f'{cls} class')
        ax.set_xlabel('Mean Predicted Probability')
        ax.set_ylabel('Fraction of Positives')
        ax.legend()

    plt.suptitle('Calibration Curves (Final Model)', fontsize=14, y=1.01)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'calibration_curves_final.png', dpi=150)
    plt.show()
else:
    print('Model bundle not yet available.')

## 8. Classification Report Summary

In [ ]:
from sklearn.metrics import classification_report

if bundle_path.exists() and data_path.exists():
    print('=== Final Model -- Classification Report (Test Set) ===')
    print(classification_report(y_test, y_pred, target_names=CLASS_NAMES))
else:
    print('Model bundle not yet available.')

## 9. Experiment Summary

In [ ]:
print('=== EXPERIMENT SUMMARY ===')
if not ranking_df.empty:
    best = ranking_df.iloc[0]
    print(f'Best base model:   {best["model_name"]} (F1_macro={best["f1_macro"]:.4f})')
    print(f'Worst base model:  {ranking_df.iloc[-1]["model_name"]} (F1_macro={ranking_df.iloc[-1]["f1_macro"]:.4f})')
    print(f'F1 range:          {ranking_df["f1_macro"].min():.4f} - {ranking_df["f1_macro"].max():.4f}')

if final_df is not None and 'f1_macro' in final_df.columns:
    final_best = final_df.loc[final_df['f1_macro'].idxmax()]
    print(f'Best final model:  {final_best["model_name"]} ({final_best.get("stage","")})')
    print(f'Best final F1:     {final_best["f1_macro"]:.4f}')